# Resumen Observacional: Beta=0 vs Beta óptima

Compara, para los 8 experimentos "Observacional" (4 ruidos aditivos + 4 multiplicativos), el modelo base
(`Beta=0`, sin término HSIC) frente al modelo con la `Beta` óptima elegida automáticamente según un
criterio multi-métrica (`MAE Z`, `HSIC(Z,X)`, `HSIC(Z,Y)`, `RF Acc`), y comprueba con un test de
Wilcoxon pareado si las mejoras son significativas.

Todo el análisis se repite para dos tamaños muestrales, `N=50` y `N=100`, para ver si las conclusiones
son estables al cambiar N.


In [1]:
import pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
NOTEBOOKS_DIR = REPO_ROOT / "notebooks"

ALPHA = 0.05
METRICAS = ["MAE Z", "HSIC(Z,X)", "HSIC(Z,Y)", "RF Acc"]  # menor es mejor en las 4


## Definición de los 8 experimentos

In [2]:
EXPERIMENTOS = {
    "Aditivo Gaussiano": {
        "path": NOTEBOOKS_DIR / "Experimento1/Observacional/tablas/aditivo_gausian.csv",
        "incompleto": False,
    },
    "Aditivo Exponencial": {
        "path": NOTEBOOKS_DIR / "Experimento1/Observacional/tablas/aditivo_exponential.csv",
        "incompleto": False,
    },
    "Aditivo Gamma": {
        "path": NOTEBOOKS_DIR / "Experimento1/Observacional/tablas/aditivo_gamma.csv",
        "incompleto": False,
    },
    "Aditivo Uniforme": {
        "path": NOTEBOOKS_DIR / "Experimento1/Observacional/tablas/aditivo_uniform.csv",
        "incompleto": False,
    },
    "Multiplicativo Gaussiano": {
        "path": NOTEBOOKS_DIR / "Experimento2/Observacional/tablas/multiplicativo_gausian.csv",
        "incompleto": False,
    },
    "Multiplicativo Exponencial": {
        "path": NOTEBOOKS_DIR / "Experimento2/Observacional/tablas/multiplicativo_exponencial.csv",
        "incompleto": False,
    },
    "Multiplicativo Gamma": {
        "path": NOTEBOOKS_DIR / "Experimento2/Observacional/tablas/multiplicativo_gamma.csv",
        "incompleto": False,
    },
    "Multiplicativo Uniforme": {
        "path": NOTEBOOKS_DIR / "Experimento2/Observacional/tablas/multiplicativo_uniforme.csv",
        "incompleto": False,
    },
}

for nombre, info in EXPERIMENTOS.items():
    assert info["path"].exists(), f"No existe: {info['path']}"


## Función de selección de la Beta óptima

In [3]:
def beta_optima(df: pd.DataFrame, n_filter: int, metrics=METRICAS):
    """Selecciona la Beta óptima (Beta != 0) por prioridad estricta: RF Acc -> MMD -> MAE Z -> Beta.

    Beta=0 se usa solo como referencia (baseline) y nunca puede ser el resultado.

    1) Filtra N == n_filter, promedia cada métrica por Beta (sobre las seeds disponibles) y
       descarta Beta=0 del conjunto de candidatas.
    2) Ordena las Beta > 0 de menor a mayor RF Acc media; los empates se rompen, en orden, por
       menor MMD media, luego por menor MAE Z media y, si aún persiste el empate, por la Beta
       más pequeña. Se elige la primera de la lista ordenada.

    Devuelve (baseline: Series, beta_opt: float, valores_opt: Series, candidatas: DataFrame,
    criterio: str), donde `candidatas` es el ranking completo (todas las Beta > 0, columnas
    RF Acc/MMD/MAE Z) usado para el desempate.
    """
    df_n = df[df["N"] == n_filter]
    columnas = sorted(set(metrics) | {"MMD"})
    media_por_beta = df_n.groupby("Beta")[columnas].mean()

    baseline = media_por_beta.loc[0.0]
    media_no_cero = media_por_beta.drop(index=0.0)

    orden_desempate = ["RF Acc", "MMD", "MAE Z"]
    candidatas = media_no_cero[orden_desempate].reset_index().sort_values(
        by=orden_desempate + ["Beta"],
    ).reset_index(drop=True)

    ganadora = candidatas.iloc[0]
    beta_opt = float(ganadora["Beta"])
    valores_opt = media_por_beta.loc[beta_opt]

    eps = 1e-9
    n_tied_rf = int((abs(media_no_cero["RF Acc"] - ganadora["RF Acc"]) < eps).sum())
    if n_tied_rf == 1:
        criterio = "menor RF Acc media"
    else:
        empatadas_rf = media_no_cero[abs(media_no_cero["RF Acc"] - ganadora["RF Acc"]) < eps]
        n_tied_mmd = int((abs(empatadas_rf["MMD"] - ganadora["MMD"]) < eps).sum())
        if n_tied_mmd == 1:
            criterio = f"empate en RF Acc media entre {n_tied_rf} betas, desempate por menor MMD"
        else:
            empatadas_mmd = empatadas_rf[abs(empatadas_rf["MMD"] - ganadora["MMD"]) < eps]
            n_tied_mae = int((abs(empatadas_mmd["MAE Z"] - ganadora["MAE Z"]) < eps).sum())
            if n_tied_mae == 1:
                criterio = f"empate en RF Acc y MMD medias entre {n_tied_mmd} betas, desempate por menor MAE Z"
            else:
                criterio = f"empate en RF Acc, MMD y MAE Z medias entre {n_tied_mae} betas, desempate por la Beta más pequeña"

    return baseline, beta_opt, valores_opt, candidatas, criterio


## Función de test de Wilcoxon pareado

Para cada experimento y cada métrica, se comparan los valores por seed en `Beta=0` frente a
`Beta=beta_optima`, emparejados por `Seed`. Test unidireccional (`alternative='less'`): H0 = no hay
diferencia; H1 = el valor con la beta óptima es menor (mejor) que con beta=0.


In [4]:
def wilcoxon_experimento(df: pd.DataFrame, beta_opt: float, n_filter: int, metrics=METRICAS):
    """Wilcoxon signed-rank pareado (por Seed) entre Beta=0 y Beta=beta_opt, por métrica.

    H0: no hay diferencia. H1 (alternative='less'): el valor en beta_opt es menor (mejor) que en Beta=0.
    Devuelve (p_valores: dict metrica->p, n_pares: int).
    """
    df_n = df[df["N"] == n_filter]
    base = df_n[df_n["Beta"] == 0.0].set_index("Seed")[list(metrics)]
    opt = df_n[df_n["Beta"] == beta_opt].set_index("Seed")[list(metrics)]
    seeds_comunes = sorted(set(base.index) & set(opt.index))
    base = base.loc[seeds_comunes]
    opt = opt.loc[seeds_comunes]

    p_valores = {}
    for m in metrics:
        try:
            _, p = wilcoxon(opt[m].values, base[m].values, alternative="less")
        except ValueError:
            p = float("nan")
        p_valores[m] = p
    return p_valores, len(seeds_comunes)


## Cálculo (tabla resumen + p-valores) para un N dado

In [5]:
def analizar_n(n_filter: int):
    """Ejecuta beta_optima + wilcoxon_experimento para los 8 experimentos, a un N fijo.

    Devuelve (resumen: DataFrame, p_values: DataFrame, diagnostico: dict[str, DataFrame]).
    """
    filas_resumen = []
    filas_p = []
    diagnostico = {}

    for nombre, info in EXPERIMENTOS.items():
        df = pd.read_csv(info["path"])
        baseline, beta_opt, valores_opt, candidatas, criterio = beta_optima(df, n_filter=n_filter)
        diagnostico[nombre] = candidatas

        nota = "\u26a0 datos parciales" if info["incompleto"] else ""

        filas_resumen.append({
            "Experimento": nombre,
            "beta_optima": beta_opt,
            "MAE(Z) beta=0": baseline["MAE Z"],
            "MAE(Z) beta_optima": valores_opt["MAE Z"],
            "HSIC(Z,X) beta=0": baseline["HSIC(Z,X)"],
            "HSIC(Z,X) beta_optima": valores_opt["HSIC(Z,X)"],
            "HSIC(Z,Y) beta=0": baseline["HSIC(Z,Y)"],
            "HSIC(Z,Y) beta_optima": valores_opt["HSIC(Z,Y)"],
            "RF Acc beta=0": baseline["RF Acc"],
            "RF Acc beta_optima": valores_opt["RF Acc"],
            "Criterio": criterio,
            "Nota": nota,
        })

        p_valores, n_pares = wilcoxon_experimento(df, beta_opt, n_filter=n_filter)
        filas_p.append({"Experimento": nombre, **p_valores, "n_seeds_pareadas": n_pares})

    resumen = pd.DataFrame(filas_resumen)
    p_values = pd.DataFrame(filas_p).set_index("Experimento")
    return resumen, p_values, diagnostico


resumen_50, p_values_50, diagnostico_50 = analizar_n(50)
resumen_100, p_values_100, diagnostico_100 = analizar_n(100)


## Funciones de formato y resaltado en negrita

In [6]:
COLUMNA_A_METRICA = {
    "MAE(Z) beta_optima": "MAE Z",
    "HSIC(Z,X) beta_optima": "HSIC(Z,X)",
    "HSIC(Z,Y) beta_optima": "HSIC(Z,Y)",
    "RF Acc beta_optima": "RF Acc",
}


def formatear(resumen: pd.DataFrame):
    cols_num = [c for c in resumen.columns if c not in ("Experimento", "beta_optima", "Criterio", "Nota")]
    fmt = resumen.copy()
    fmt[cols_num] = fmt[cols_num].round(5)
    fmt["beta_optima"] = fmt["beta_optima"].round(2)
    return fmt


def tabla_con_negrita(resumen: pd.DataFrame, p_values: pd.DataFrame):
    """Tabla resumen con las celdas 'beta_optima' en negrita cuando su p-valor (Wilcoxon) < ALPHA.

    Solo aplica en la visualización del notebook (un CSV plano no admite negrita).
    """
    fmt = formatear(resumen)

    def resaltar(row):
        p_exp = p_values.loc[row["Experimento"]]
        estilos = []
        for col in row.index:
            metrica = COLUMNA_A_METRICA.get(col)
            if metrica is not None and pd.notna(p_exp[metrica]) and p_exp[metrica] < ALPHA:
                estilos.append("font-weight: bold")
            else:
                estilos.append("")
        return estilos

    return fmt.style.apply(resaltar, axis=1)


## N = 50

In [7]:
tabla_con_negrita(resumen_50, p_values_50)


,Experimento,beta_optima,MAE(Z) beta=0,MAE(Z) beta_optima,"HSIC(Z,X) beta=0","HSIC(Z,X) beta_optima","HSIC(Z,Y) beta=0","HSIC(Z,Y) beta_optima",RF Acc beta=0,RF Acc beta_optima,Criterio,Nota
0,Aditivo Gaussiano,0.900000,1.152110,1.554610,0.058760,0.123470,0.115160,0.238070,0.596250,0.587500,menor RF Acc media,
1,Aditivo Exponencial,0.900000,0.888200,0.974520,0.049410,0.046830,0.115100,0.112180,0.687500,0.675000,menor RF Acc media,
2,Aditivo Gamma,0.900000,1.160910,1.499200,0.170450,0.232710,0.218600,0.289390,0.607500,0.595000,menor RF Acc media,
3,Aditivo Uniforme,0.300000,0.989990,1.027680,0.045490,0.046140,0.058510,0.078530,0.605000,0.570000,menor RF Acc media,
4,Multiplicativo Gaussiano,0.400000,1.016720,0.912500,0.151350,0.133210,0.073290,0.074160,0.622500,0.615000,menor RF Acc media,
5,Multiplicativo Exponencial,0.100000,0.727840,0.684620,0.218180,0.159620,0.059910,0.059490,0.738750,0.713750,menor RF Acc media,
6,Multiplicativo Gamma,0.600000,1.000840,0.757680,0.215460,0.157130,0.136270,0.097640,0.650000,0.618750,menor RF Acc media,
7,Multiplicativo Uniforme,0.400000,0.762230,0.754080,0.110800,0.100610,0.085910,0.088290,0.616250,0.606250,menor RF Acc media,


In [8]:
p_values_50.round(4)


,MAE Z,"HSIC(Z,X)","HSIC(Z,Y)",RF Acc,n_seeds_pareadas
Experimento,,,,,
Aditivo Gaussiano,1.0000,0.9998,0.9999,0.2994,20
Aditivo Exponencial,0.9953,0.4347,0.3506,0.1811,20
Aditivo Gamma,0.9972,0.9968,0.9867,0.1741,20
Aditivo Uniforme,0.6892,0.1471,0.9285,0.0288,20
Multiplicativo Gaussiano,0.0527,0.0319,0.8919,0.2999,20
Multiplicativo Exponencial,0.0004,0.0004,0.3238,0.0023,20
Multiplicativo Gamma,0.0010,0.0379,0.0242,0.0298,20
Multiplicativo Uniforme,0.3108,0.1313,0.7147,0.2962,20


## N = 100

In [9]:
tabla_con_negrita(resumen_100, p_values_100)


,Experimento,beta_optima,MAE(Z) beta=0,MAE(Z) beta_optima,"HSIC(Z,X) beta=0","HSIC(Z,X) beta_optima","HSIC(Z,Y) beta=0","HSIC(Z,Y) beta_optima",RF Acc beta=0,RF Acc beta_optima,Criterio,Nota
0,Aditivo Gaussiano,0.800000,0.954930,1.093740,0.039320,0.043420,0.067360,0.105170,0.601250,0.568750,menor RF Acc media,
1,Aditivo Exponencial,0.400000,0.816590,0.781290,0.057420,0.045080,0.088230,0.064530,0.635000,0.640000,menor RF Acc media,
2,Aditivo Gamma,0.600000,0.835690,0.981160,0.105580,0.125370,0.099300,0.128590,0.617500,0.591250,menor RF Acc media,
3,Aditivo Uniforme,0.500000,0.956230,0.971020,0.032190,0.035950,0.047970,0.059750,0.585000,0.567500,menor RF Acc media,
4,Multiplicativo Gaussiano,0.200000,0.797320,0.793110,0.081560,0.083010,0.072960,0.068310,0.598750,0.592500,menor RF Acc media,
5,Multiplicativo Exponencial,0.600000,0.696900,0.645970,0.212110,0.114760,0.064740,0.042950,0.731250,0.707500,"empate en RF Acc media entre 2 betas, desempate por menor MMD",
6,Multiplicativo Gamma,0.300000,0.890710,0.715230,0.188120,0.120670,0.131720,0.108840,0.636250,0.622500,menor RF Acc media,
7,Multiplicativo Uniforme,0.700000,0.714440,0.723940,0.090790,0.088930,0.097780,0.087000,0.606250,0.597500,menor RF Acc media,


In [10]:
p_values_100.round(4)


,MAE Z,"HSIC(Z,X)","HSIC(Z,Y)",RF Acc,n_seeds_pareadas
Experimento,,,,,
Aditivo Gaussiano,1.0000,0.8529,1.0000,0.0287,20
Aditivo Exponencial,0.0715,0.0007,0.0053,0.6590,20
Aditivo Gamma,0.9982,0.8159,0.8919,0.0319,20
Aditivo Uniforme,0.8919,0.9552,0.9780,0.0674,20
Multiplicativo Gaussiano,0.3643,0.0768,0.0164,0.2838,20
Multiplicativo Exponencial,0.0001,0.0001,0.0005,0.0648,20
Multiplicativo Gamma,0.0014,0.0042,0.0348,0.3487,20
Multiplicativo Uniforme,0.9552,0.2045,0.0060,0.3780,20


## Comparación N=50 vs N=100

Beta óptima elegida y número de métricas significativas (p < 0.05) en cada N, para ver de un vistazo
si las conclusiones cambian con el tamaño muestral.


In [11]:
comparacion = resumen_50[["Experimento", "beta_optima"]].rename(columns={"beta_optima": "beta_optima N=50"})
comparacion = comparacion.merge(
    resumen_100[["Experimento", "beta_optima"]].rename(columns={"beta_optima": "beta_optima N=100"}),
    on="Experimento",
)
comparacion["metricas_sig N=50"] = comparacion["Experimento"].map((p_values_50[METRICAS] < ALPHA).sum(axis=1))
comparacion["metricas_sig N=100"] = comparacion["Experimento"].map((p_values_100[METRICAS] < ALPHA).sum(axis=1))
comparacion


,Experimento,beta_optima N=50,beta_optima N=100,metricas_sig N=50,metricas_sig N=100
0,Aditivo Gaussiano,0.9,0.8,0,1
1,Aditivo Exponencial,0.9,0.4,0,2
2,Aditivo Gamma,0.9,0.6,0,1
3,Aditivo Uniforme,0.3,0.5,1,0
4,Multiplicativo Gaussiano,0.4,0.2,1,1
5,Multiplicativo Exponencial,0.1,0.6,3,3
6,Multiplicativo Gamma,0.6,0.3,4,3
7,Multiplicativo Uniforme,0.4,0.7,0,1


## Diagnóstico de candidatas (opcional)

Para inspeccionar, por experimento y N, qué betas fueron candidatas (óptimas para al menos una métrica)
y por qué se eligió la ganadora, descomenta y ejecuta:


In [12]:
# for nombre, tabla in diagnostico_50.items():
#     print(nombre, "(N=50)")
#     display(tabla)
# for nombre, tabla in diagnostico_100.items():
#     print(nombre, "(N=100)")
#     display(tabla)


## Guardar CSVs resumen

In [13]:
OUT_PATH_50 = NOTEBOOKS_DIR / "tablas" / "resumen_observacional_beta_n50.csv"
OUT_PATH_100 = NOTEBOOKS_DIR / "tablas" / "resumen_observacional_beta_n100.csv"

resumen_50.to_csv(OUT_PATH_50, index=False)
resumen_100.to_csv(OUT_PATH_100, index=False)

print(f"Guardado en: {OUT_PATH_50}")
print(f"Guardado en: {OUT_PATH_100}")


Guardado en: c:\Users\aarna\Desktop\clau\TFM Claudia\kacgm-hsic\notebooks\tablas\resumen_observacional_beta_n50.csv
Guardado en: c:\Users\aarna\Desktop\clau\TFM Claudia\kacgm-hsic\notebooks\tablas\resumen_observacional_beta_n100.csv
